# 🧠 SVOMPTR-9B: Master Neural Distillation & MoE Training

This notebook covers:
- 🛡️ Google Drive Integration (`svomptr_brain`)
- 🗃️ Massive 5M Data Generation via vLLM
- 🧺 Fast Knowledge Distillation & DOP Alignment
- 🔄 Auto-resume & Idle Prevention
- 🚀 LocalTunnel ML API Deployment

### 🛡️ 1. Prevent Idle Timeout
Press `Ctrl+Shift+I` (or `Cmd+Option+I` on Mac) to open browser Developer Tools. Go to the **Console** tab, paste the following code, and press Enter. This will keep Colab alive during long training sessions.

```javascript
function KeepClicking(){
  console.log("Keeping Colab active...");
  document.querySelector("colab-toolbar-button").click();
}
setInterval(KeepClicking, 60000);
```

In [ ]:
# 🛡️ 2. Drive Mount & Brain Initialization
from google.colab import drive
import os

drive.mount('/content/drive')
brain_dir = '/content/drive/MyDrive/svomptr_brain'

# Create necessary directories
directories = ['datasets', 'checkpoints', 'weights', 'dop_alignment']
for d in directories:
    os.makedirs(os.path.join(brain_dir, d), exist_ok=True)

print(f"✅ SVOMPTR Brain active at: {brain_dir}")

In [ ]:
# ⚙️ 3. Install Core Dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate bitsandbytes datasets faiss-gpu sentence-transformers fastapi uvicorn pydantic pyngrok vllm trl

In [ ]:
# 📂 4. Project Setup
import sys
import os
repo_path = '/content/svomptr-project'
if not os.path.exists(repo_path):
    print("Cloning SVOMPTR repository to /content/svomptr-project...")
    !git clone https://github.com/kkomyoeminaung/svomptr-9b-moe-model-.git /content/svomptr-project
sys.path.append(repo_path)

### 🗃️ 5. Massive Synthetic Data Generation (vLLM for Speed)
Generates millions of structured SVOMPTR English-Myanmar pairs rapidly. Resumes automatically if interrupted.

In [ ]:
%%writefile generate_data.py
import os
import json
import time
try:
    brain_dir = '/content/drive/MyDrive/svomptr_brain'
    dataset_file = os.path.join(brain_dir, 'datasets', 'synthetic_5M.jsonl')
    teacher_model_id = "Qwen/Qwen2.5-7B-Instruct"
    
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU available for vLLM.")
    from vllm import LLM, SamplingParams
    print(f"Initializing vLLM with {teacher_model_id}...")
    llm = LLM(model=teacher_model_id, dtype="bfloat16", max_model_len=4096, enable_prefix_caching=True)
    sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=300)
    USE_VLLM = True
except Exception as e:
    print(f"vLLM initialization failed ({e}). Falling back to Hugging Face transformers.")
    from transformers import pipeline
    pipe = pipeline('text-generation', model=teacher_model_id, device_map='auto')
    USE_VLLM = False

generated_count = 0
if os.path.exists(dataset_file):
    with open(dataset_file, 'r', encoding='utf-8') as f:
        generated_count = sum(1 for _ in f)
print(f"Resuming generation from {generated_count} synthesized items...")

TARGET_COUNT = 5_000_000
BATCH_SIZE = 5000 if USE_VLLM else 10 # Drastically reduce batch size for CPU/HF fallback

base_prompt = """
Generate 1 complex English sentence focusing on advanced concepts (science, technology, philosophy, daily life).
Provide:
1. The English sentence
2. A highly accurate Myanmar translation
3. SVOMPTR structure analysis (Subject, Verb, Object, Modifier, Place, Time, Reason)

Output strictly as a single JSON object in this exact format, with no markdown or backticks:
{"en": "...", "my": "...", "svomptr_structure": {"S": "...", "V": "...", "O": "...", "M": "...", "P": "...", "T": "...", "R": "..."}}
"""

while generated_count < TARGET_COUNT:
    prompts = [base_prompt for _ in range(BATCH_SIZE)]
    t0 = time.time()
    
    if USE_VLLM:
        outputs_raw = llm.generate(prompts, sampling_params, use_tqdm=True)
        outputs = [out.outputs[0].text.strip() for out in outputs_raw]
    else:
        outputs_raw = pipe(prompts, max_new_tokens=300, temperature=0.7, top_p=0.9, do_sample=True)
        outputs = [out[0]['generated_text'][len(base_prompt):].strip() for out in outputs_raw]
    
    with open(dataset_file, 'a', encoding='utf-8') as f:
        for text in outputs:
            # Basic robust JSON cleaning
            if text.startswith('```json'):
                text = text.replace('```json', '').replace('```', '').strip()
            elif text.startswith('```'):
                text = text.replace('```', '').strip()
                
            if text.startswith('{') and text.endswith('}'):
                # Remove newlines inside JSON to keep it JSONLines compliant
                cleaned_text = ' '.join(text.splitlines())
                f.write(cleaned_text + '\n')
                generated_count += 1
                
    print(f"[Distillation Data Engine] Total: {generated_count}/{TARGET_COUNT}. Batch pace: {BATCH_SIZE / (time.time()-t0):.2f} iter/s")
    if generated_count >= TARGET_COUNT:
        break


In [ ]:
# !python generate_data.py

### 🧺 6. Distillation & DOP Alignment Training
Train the Chat Expert student model using the synthetic dataset. Auto-saves to Drive and resumes on interruption.

In [ ]:
%%writefile train_distillation.py
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset

brain_dir = '/content/drive/MyDrive/svomptr_brain'
dataset_file = os.path.join(brain_dir, 'datasets', 'synthetic_5M.jsonl')
ckpt_dir = os.path.join(brain_dir, 'checkpoints', 'chat_expert')
final_weight_dir = os.path.join(brain_dir, 'weights', 'chat_expert_final')
dop_dir = os.path.join(brain_dir, 'dop_alignment')

# Create DOP alignment metadata tracker
os.makedirs(dop_dir, exist_ok=True)

print("[Distillation] Loading Dataset...")
if not os.path.exists(dataset_file):
    print(f"Dataset not found at {dataset_file}.")
    print("Falling back to basic repository data via huggingface or local file.")
    # Update this with your path if needed
    dataset_file = '/content/svomptr-project/data/svomptr_chat_pairs.jsonl' 

student_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"[Distillation] Initializing Student Model: {student_model_id}")
tokenizer = AutoTokenizer.from_pretrained(student_model_id)
model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

def formatting_func(example):
    if 'text' in example:
        return {'text': example['text']}
    # Standardize format for Causal LM training
    try:
        en_text = example.get('en', '')
        my_text = example.get('my', '')
        struct = example.get('svomptr_structure', {})
        struct_str = f"S:{struct.get('S','')}, V:{struct.get('V','')}, O:{struct.get('O','')}"
        
        prompt = f"<|im_start|>user\nTranslate and analyze: {en_text}<|im_end|>\n<|im_start|>model\n"
        response = f"Translation: {my_text}\nStructure: {struct_str}<|im_end|>"
        return {"text": prompt + response}
    except Exception:
        # Fallback for plain conversation data
        return {"text": f"<|im_start|>user\n{example.get('prompt', '')}<|im_end|>\n<|im_start|>model\n{example.get('response', '')}<|im_end|>"}

try:
    raw_dataset = load_dataset("json", data_files=dataset_file, split="train")
    dataset = raw_dataset.map(formatting_func, remove_columns=raw_dataset.column_names)
except Exception as e:
    print(f"Dataset loading failed: {e}")
    exit(1)

training_args = TrainingArguments(
    output_dir=ckpt_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    save_strategy="steps",
    save_steps=500,
    logging_steps=50,
    bf16=True,
    save_total_limit=3, # Retain last 3 checkpoints
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    # data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

last_checkpoint = None
if os.path.isdir(ckpt_dir):
    checkpoints = [os.path.join(ckpt_dir, d) for d in os.listdir(ckpt_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"[Distillation] Resuming from existing checkpoint: {last_checkpoint}")

print("[Distillation] Initiating DOP Alignment Sequence...")
trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"[Distillation] Complete! Saving optimized weights to {final_weight_dir}")
model.save_pretrained(final_weight_dir)
tokenizer.save_pretrained(final_weight_dir)

# Save DOP signature
with open(os.path.join(dop_dir, "alignment_signature.txt"), "w") as f:
    f.write("DOP Alignment: SUCCESS\nModel: Qwen2.5-1.5B (Distilled)\n")


In [ ]:
# !python train_distillation.py

### 🤖 6.5. Train Router & Sub-Experts (Full MoE)
Once the main chat expert is distilled, train the semantic router and the sub-domain specialists.

In [ ]:
import sys
if '/content/svomptr-project' not in sys.path:
    sys.path.insert(0, '/content/svomptr-project')

from svomptr_moe.train_sub_experts import train_domain_experts
from svomptr_moe.train_router import train_router

# Set brain path to ensure weights map to Google Drive
import os
os.environ['SVOMPTR_BRAIN_PATH'] = '/content/drive/MyDrive/svomptr_brain'

print("Training Sub-Experts...")
# train_domain_experts()
print("Training Router...")
# train_router()


### 🚀 7. Run ML API (LocalTunnel)
Launch the FastAPI backend serving the weights from `svomptr_brain`.

In [ ]:
!npm install -g localtunnel
import subprocess
import time

print("Starting SVOMPTR FastAPI server...")
# Make sure to set SVOMPTR_BRAIN_PATH inside your repo if you want it to load weights from Drive
import os
os.environ["SVOMPTR_BRAIN_PATH"] = "/content/drive/MyDrive/svomptr_brain"

api_proc = subprocess.Popen(["uvicorn", "ml_api:app", "--host", "0.0.0.0", "--port", "8000", "--reload", "--app-dir", "/content/svomptr-project"])
time.sleep(5)

print("Starting LocalTunnel... Copy the URL below and set VITE_ML_API_URL and ML_API_URL in your Next.js/Vite frontend!")
!lt --port 8000 --subdomain svogateway9b